# CAD Detection — EfficientNet-B0 (PyTorch)
**Before running:**
1. Runtime → Change runtime type → T4 GPU
2. Get your Kaggle API token: kaggle.com → Settings → API → Create New Token → save as `kaggle.json`
3. Run all cells — cell 2 will prompt you to upload `kaggle.json`

In [ ]:
!pip install kaggle -q

from google.colab import files
files.upload()  # select your kaggle.json when prompted

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d danialsharifrazi/cad-cardiac-mri-dataset
!unzip -q cad-cardiac-mri-dataset.zip -d /content/dataset
print("Dataset ready.")

In [ ]:
import os, sys, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| Device:', DEVICE)

In [ ]:
DATASET_PATH   = '/content/dataset'
MODEL_OUT      = '/content/cad_model.pth'
IMG_SIZE       = 224
BATCH_SIZE     = 32
EPOCHS_PHASE1  = 15
EPOCHS_PHASE2  = 10

print(os.listdir(DATASET_PATH))

In [ ]:
%%writefile /content/preprocess.py
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from PIL import Image

IMG_SIZE      = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

infer_transform = val_transform


class MRIDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("L")
        arr = np.array(img)
        if arr.max() == 0:
            arr = np.full_like(arr, 128)
        arr = _clahe.apply(arr)
        img = Image.fromarray(arr)
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.float32)


def get_patients(dataset_path):
    patient_dirs, labels = [], []
    for label, class_name in enumerate(["Normal", "Sick"]):
        class_dir = os.path.join(dataset_path, class_name)
        for name in sorted(os.listdir(class_dir)):
            path = os.path.join(class_dir, name)
            if os.path.isdir(path):
                patient_dirs.append(path)
                labels.append(label)
    return patient_dirs, labels


def patient_to_slices(patient_dirs, labels):
    paths, slice_labels = [], []
    for patient_dir, label in zip(patient_dirs, labels):
        for series_name in sorted(os.listdir(patient_dir)):
            series_path = os.path.join(patient_dir, series_name)
            if not os.path.isdir(series_path):
                continue
            for f in sorted(os.listdir(series_path)):
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                    paths.append(os.path.join(series_path, f))
                    slice_labels.append(label)
    return paths, slice_labels

In [ ]:
%%writefile /content/model_arch.py
import torch.nn as nn
import torchvision.models as models


def build_model():
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
    model = models.efficientnet_b0(weights=weights)

    for param in model.parameters():
        param.requires_grad = False

    in_features = model.classifier[1].in_features  # 1280
    model.classifier = nn.Sequential(
        nn.BatchNorm1d(in_features),
        nn.Dropout(0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, 1),
        nn.Sigmoid(),
    )
    return model


def unfreeze_top_layers(model, n_blocks=3):
    for block_idx in range(9 - n_blocks, 9):
        for param in model.features[block_idx].parameters():
            param.requires_grad = True

In [ ]:
%%writefile /content/train.py
import os
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

from model_arch import build_model, unfreeze_top_layers
from preprocess import (
    MRIDataset, get_patients, patient_to_slices,
    train_transform, val_transform,
)

DATASET_PATH    = r"C:\Users\yycha\Downloads\dataset"
MODEL_SAVE_PATH = "model/cad_model.pth"
BATCH_SIZE    = 32
EPOCHS_PHASE1 = 15
EPOCHS_PHASE2 = 10
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def train_epoch(model, loader, optimizer, criterion, device, class_weights):
    model.train()
    total_loss = correct = total = 0
    all_probs, all_labels = [], []

    w = torch.tensor(class_weights, dtype=torch.float32).to(device)

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        probs = model(imgs).squeeze(1)

        weights = torch.where(labels == 1, w[1], w[0])
        loss = (criterion(probs, labels) * weights).mean()

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        preds = (probs > 0.5).float()
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
        all_probs.extend(probs.detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / total, correct / total, auc


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = correct = total = 0
    all_probs, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            probs = model(imgs).squeeze(1)
            loss  = criterion(probs, labels).mean()

            total_loss += loss.item() * imgs.size(0)
            preds = (probs > 0.5).float()
            correct += (preds == labels).sum().item()
            total   += imgs.size(0)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / total, correct / total, auc


def run_phase(model, train_loader, val_loader, optimizer, scheduler,
              criterion, epochs, device, class_weights, model_path):
    best_val_loss = float("inf")
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
               "train_auc": [], "val_auc": []}
    patience_counter = 0
    patience = 5

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc, tr_auc = train_epoch(
            model, train_loader, optimizer, criterion, device, class_weights
        )
        vl_loss, vl_acc, vl_auc = eval_epoch(model, val_loader, criterion, device)
        scheduler.step(vl_loss)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)
        history["train_auc"].append(tr_auc)
        history["val_auc"].append(vl_auc)

        print(
            f"Epoch {epoch:02d}  "
            f"Loss {tr_loss:.4f}/{vl_loss:.4f}  "
            f"Acc {tr_acc:.4f}/{vl_acc:.4f}  "
            f"AUC {tr_auc:.4f}/{vl_auc:.4f}"
        )

        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            torch.save(model.state_dict(), model_path)
            print(f"  -> saved best model (val_loss={vl_loss:.4f})")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch}")
                break

    model.load_state_dict(torch.load(model_path, map_location=device))
    return history


def full_evaluation(model, loader, device, save_dir="static"):
    from sklearn.metrics import (
        confusion_matrix, roc_curve, auc,
        precision_recall_curve, average_precision_score,
    )
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            probs = model(imgs).squeeze(1)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds = (all_probs >= 0.5).astype(int)

    cm = confusion_matrix(all_labels, preds)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    precision   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = (2 * precision * sensitivity / (precision + sensitivity)
          if (precision + sensitivity) > 0 else 0.0)

    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    roc_auc     = auc(fpr, tpr)
    prec_c, rec_c, _ = precision_recall_curve(all_labels, all_probs)
    ap = average_precision_score(all_labels, all_probs)

    print(f"\n=== Full Evaluation ===")
    print(f"Sensitivity (Recall): {sensitivity:.4f}")
    print(f"Specificity:          {specificity:.4f}")
    print(f"Precision:            {precision:.4f}")
    print(f"F1 Score:             {f1:.4f}")
    print(f"ROC AUC:              {roc_auc:.4f}")
    print(f"Avg Precision (PR):   {ap:.4f}")
    print(f"Confusion Matrix  →  TN={tn}  FP={fp}  |  FN={fn}  TP={tp}")

    os.makedirs(save_dir, exist_ok=True)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Normal", "Sick"])
    ax.set_yticklabels(["Normal", "Sick"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "confusion_matrix.png"), dpi=100)
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("CAD Detection — Model Evaluation")
    axes[0].plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.4f}")
    axes[0].plot([0, 1], [0, 1], "k--", lw=1)
    axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title("ROC Curve"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(rec_c, prec_c, lw=2, label=f"AP = {ap:.4f}")
    axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
    axes[1].set_title("Precision-Recall Curve"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "roc_pr_curves.png"), dpi=100)
    plt.show()

    return {
        "sensitivity": sensitivity, "specificity": specificity,
        "precision": precision, "f1": f1, "roc_auc": roc_auc, "ap": ap,
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
    }


def plot_history(h1, h2, save_path="static/training_history.png"):
    acc      = h1["train_acc"]  + h2["train_acc"]
    val_acc  = h1["val_acc"]    + h2["val_acc"]
    loss     = h1["train_loss"] + h2["train_loss"]
    val_loss = h1["val_loss"]   + h2["val_loss"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("CAD Detection Model — Training History")
    axes[0].plot(acc, label="Train"); axes[0].plot(val_acc, label="Val")
    axes[0].set_title("Accuracy"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(loss, label="Train"); axes[1].plot(val_loss, label="Val")
    axes[1].set_title("Loss"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=100)
    print(f"Plot saved -> {save_path}")
    plt.show()


if __name__ == "__main__":
    pass

In [ ]:
sys.path.insert(0, '/content')
from preprocess import (
    MRIDataset, get_patients, patient_to_slices,
    train_transform, val_transform,
)
from model_arch import build_model, unfreeze_top_layers
from train import train_epoch, eval_epoch, run_phase, full_evaluation, plot_history
print('Modules loaded.')

In [ ]:
print('Scanning patients...')
all_patient_dirs, all_patient_labels = get_patients(DATASET_PATH)
normal_patients = [(d, l) for d, l in zip(all_patient_dirs, all_patient_labels) if l == 0]
sick_patients   = [(d, l) for d, l in zip(all_patient_dirs, all_patient_labels) if l == 1]
print(f'Total patients: {len(all_patient_dirs)}  Normal: {len(normal_patients)}  Sick: {len(sick_patients)}')

rng = random.Random(42)
rng.shuffle(normal_patients)
rng.shuffle(sick_patients)

N_VAL = N_TEST = 2
val_patients   = normal_patients[:N_VAL]             + sick_patients[:N_VAL]
test_patients  = normal_patients[N_VAL:N_VAL+N_TEST] + sick_patients[N_VAL:N_VAL+N_TEST]
train_patients = normal_patients[N_VAL+N_TEST:]      + sick_patients[N_VAL+N_TEST:]
print(f'Patients — Train: {len(train_patients)}  Val: {len(val_patients)}  Test: {len(test_patients)}')

train_dirs, train_lbls = zip(*train_patients)
val_dirs,   val_lbls   = zip(*val_patients)
test_dirs,  test_lbls  = zip(*test_patients)

X_train, y_train = patient_to_slices(train_dirs, train_lbls)
X_val,   y_val   = patient_to_slices(val_dirs,   val_lbls)
X_test,  y_test  = patient_to_slices(test_dirs,  test_lbls)
print(f'Slices  — Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')
print(f'Train label balance: Normal={y_train.count(0)}  Sick={y_train.count(1)}')

train_loader = DataLoader(MRIDataset(X_train, y_train, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(MRIDataset(X_val,   y_val,   val_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(MRIDataset(X_test,  y_test,  val_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

n_normal = y_train.count(0); n_sick = y_train.count(1); total = len(y_train)
class_weights = [total / (2 * n_normal), total / (2 * n_sick)]
print(f'Class weights: Normal={class_weights[0]:.3f}  Sick={class_weights[1]:.3f}')

In [ ]:
model     = build_model().to(DEVICE)
criterion = nn.BCELoss(reduction='none')
print('Model ready.')

In [ ]:
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

print('=== Phase 1: Training head ===')
h1 = run_phase(model, train_loader, val_loader, optimizer, scheduler,
               criterion, EPOCHS_PHASE1, DEVICE, class_weights, MODEL_OUT)

In [ ]:
unfreeze_top_layers(model, n_blocks=3)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

print('=== Phase 2: Fine-tuning top blocks ===')
h2 = run_phase(model, train_loader, val_loader, optimizer, scheduler,
               criterion, EPOCHS_PHASE2, DEVICE, class_weights, MODEL_OUT)

In [ ]:
tl, ta, tauc = eval_epoch(model, test_loader, criterion, DEVICE)
print(f'=== Test Results ===\nLoss: {tl:.4f}  Acc: {ta:.4f}  AUC: {tauc:.4f}')

full_evaluation(model, test_loader, DEVICE, save_dir='/content')

print(f'\nModel saved -> {MODEL_OUT}')
print('Download outputs from the Output tab on the right ->')

In [ ]:
plot_history(h1, h2, save_path='/content/training_history.png')